# Lineare Regression

Wir wollen den Zusammenhang zwischen TV-Werbebudget (x) und Verkaufszahlen (y) modellieren. Die Hypothese: Es gibt einen näherungsweise linearen Zusammenhang. Unser Modell:

$\hat{y} = w_0 + w_1 \cdot x$

Dabei ist $w_0$ der Achsenabschnitt (Bias) und $w_1$ die Steigung (Gewicht).

In der Vorlesung haben wir zwei Wege kennengelernt, die optimalen Parameter $w_0$ und $w_1$ zu finden, sodass der mittlere quadratische Fehler minimiert wird:

1. Analytische Lösung (Normalengleichung): direkte Formel, exakte Lösung.
2. Gradientenabstieg: iteratives Verfahren, das in vielen Fällen bevorzugt wird.

In diesem Notebook implementieren wir beide und vergleichen die Ergebnisse.

## Benötigte Module

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Daten laden und vorbereiten

Wir werden den kleinen Werbungsdatensatz aus der vorherigen Übung verwenden.

In [ ]:
file_dataset = 'advertising.csv'
try:
    dataset = np.loadtxt(file_dataset, delimiter=',', skiprows=1)

    # Features und Labels extrahieren
    x = dataset[:, 0]    # TV-Budget (Input / Feature)
    y = dataset[:, 3]    # Verkaufszahlen (Output / Label / Target)

    n = x.shape[0]
    print(f'Anzahl Datenpunkte: {n}')
    print(f'x (TV-Budget): min={x.min():.1f}, max={x.max():.1f}, mean={x.mean():.1f}')
    print(f'y (Verkäufe):  min={y.min():.1f}, max={y.max():.1f}, mean={y.mean():.1f}')

except FileNotFoundException:
    print('Die Datensatzdatei könnte nicht geöffnet werden.')

## Hilfsfunktionen zur Vorhersage und Verlust

In [ ]:
def predict(x, w0, w1):
    """
    Berechnet die Vorhersage des linearen Modells: ŷ = w0 + w1 * x
    
    Dank Numpy-Vektorisierung funktioniert das für einen einzelnen Wert
    (float) genauso wie für ein ganzes Array von Werten.
    
    :param x: Eingabewert(e) als `float` oder `np.ndarray`
    :param w0: Bias (Achsenabschnitt).
    :param w1: Gewicht (Steigung).
    :return: Vorhersage(n) als `float` oder `np.ndarray`.
    """
    return w0 + w1 * x


def mse_loss(x: np.ndarray, y: np.ndarray, w0, w1):
    """
    Berechnet den mittleren quadratischen Fehler (MSE) des Modells:
        L(w0, w1) = (1/n) * sum_i (y_i - ŷ_i)^2
                  = (1/n) * sum_i (y_i - w0 - w1 * x_i)^2

    :param x: Feature-Vektor als Array der Form `(n,)`.
    :param y: Label-Vektor als Array der Form `(n,)`.
    :param w0: Bias.
    :param w1: Gewicht.
    :return: MSE-Verlust.
    """
    y_pred = predict(x, w0, w1)
    residuals = y - y_pred
    return np.mean(residuals ** 2)


def print_model_info(w0, w1, mse):
    print(f'    w0 (Bias) = {w0:.4f}')
    print(f'w1 (Steigung) = {w1:.4f}')
    print(f'          MSE = {mse:.4f}')
    print(f'         RMSE = {np.sqrt(mse):.4f}')
    print(f'Interpretation: Pro 1000 USD mehr TV-Budget steigen die Verkäufe')
    print(f'                um ca. {w1 * 1000:.1f} Einheiten.')


# Test mit willkürlichen Parametern
w0_test, w1_test = 5.0, 0.05
mse_test = mse_loss(x, y, w0_test, w1_test)
print(f'MSE mit w0={w0_test}, w1={w1_test}: {mse_test:.4f}')

### Übung

Schreiben Sie eine Funktion `rmse_loss(x, y, w0, w1)`, die den Root Mean Squared Error berechnet:

$RMSE = \sqrt{MSE}$

Der RMSE hat den Vorteil, dass er in derselben Einheit wie $y$ ist (hier: Tausend Einheiten). Berechnen Sie den RMSE für die obigen Testparameter.

Erwartetes Ergebnis: ~ 3.71

In [ ]:
def rmse_loss(x: np.ndarray, y: np.ndarray, w0, w1):
    return np.sqrt(mse_loss(x, y, w0, w1))

print(f'RMSE mit w0={w0_test}, w1={w1_test}: {rmse_loss(x, y, w0_test, w1_test):.4f}')

## Analytische Lösung (Normalengelichung)

Die analytische Lösung für einfache lineare Regression (ein Feature $x$, ein Label $y$) lautet:

$w_1 = \sum{(x_i - \bar{x})(y_i - \bar{y})} / \sum{(x_i - \bar{x})^2}$

$w_0 = \bar{y} - w_1 \cdot \bar{x}$

Dies ist äquivalent zur Matrixformel $w = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$, die wir für den mehrdimensionalen Fall später verwenden werden.

Warum heißt das "Normalengleichung"? Weil man die Bedingung $dL/dw = 0$ setzt und löst – der Gradient steht senkrecht (normal) auf dem Lösungsraum.

In [ ]:
def analytical_solution(x: np.ndarray, y: np.ndarray) -> tuple:
    """
    Berechnet die optimalen Parameter w0, w1 über die Normalengleichung.
    
    :param x: Feature-Vektor als Array der Form `(n,)`.
    :param y: Label-Vektor als Array der Form `(n,)`.
    :return: Optimale Parameter als Tupel `(w0, w1)`.
    """
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    
    # Zähler: Kovarianz zwischen x und y
    covariance = np.sum((x - x_mean) * (y - y_mean))
    
    # Nenner: Varianz von x
    x_variance = np.sum((x - x_mean) ** 2)
    
    w1 = covariance / x_variance
    w0 = y_mean - w1 * x_mean
    return w0, w1


w0_analyt, w1_analyt = analytical_solution(x, y)
mse_analyt = mse_loss(x, y, w0_analyt, w1_analyt)

print('Analytische Lösung:')
print_model_info(w0_analyt, w1_analyt, mse_analyt)

## Gradientenabstieg

Beim Gradientenabstieg starten wir mit zufälligen (oder Null-)Parametern und aktualisieren sie iterativ in die Richtung des steilsten Abstiegs der Verlustfunktion.

Die partiellen Ableitungen des MSE-Verlusts sind:

$dL/dw_0 = -(2/n) \sum_{i=1}^{n}{y_i - \hat{y}_i}$

$dL/dw_1 = -(2/n) \sum_{i=1}^{n}{(y_i - \hat{y}_i) x_i}$

Die Update-Regeln (mit Lernrate $\alpha$):

$w_0 := w_0 - \alpha dL/dw_0$

$w_1 := w_1 - \alpha dL/dw_1$

Wichtig: Beide Parameter werden gleichzeitig aktualisiert - nicht nacheinander, denn dann würde das zweite Update den neuen Wert des ersten verwenden – das wäre falsch.

In [ ]:
def compute_gradients(x: np.ndarray, y: np.ndarray, w0, w1):
    """
    Berechnet die Gradienten des MSE-Verlusts bezüglich w0 und w1.
    dL/dw0 = -(2/n) * sum(y_i - ŷ_i)
    dL/dw1 = -(2/n) * sum((y_i - ŷ_i) * x_i)
    
    :param x: Feature-Vektor als Array der Form `(n,)`; normalisiert.
    :param y: Label-Vektor als Array der Form `(n,)`.
    :param w0: Aktueller Bias.
    :param w1: Aktuelles Gewicht.
    :return: Gradienten als Tupel `(dL_dw0, dL_dw1)`.
    """
    n = x.shape[0]
    y_hat = predict(x, w0, w1)
    residuals = y - y_hat
    dL_dw0 = -(2 / n) * np.sum(residuals)
    dL_dw1 = -(2 / n) * np.sum(residuals * x)
    return dL_dw0, dL_dw1

## Feature-Normalisierung

Unser TV-Budget liegt im Bereich [8, 281]. Das führt dazu, dass der Gradient bezüglich $w_1$ sehr viel grösser ist als der Gradient bezüglich $w_0$ (weil $x_i$ in $dL/dw1$ als Faktor auftaucht). Die Verlustlandschaft ist dann sehr "elliptisch" – flach in eine Richtung, steil in die andere.

Das macht es schwer, eine Lernrate zu wählen, die in beiden Richtungen gut funktioniert: zu gross führt zu Divergenz, zu klein - zu langsame Konvergenz.

Lösung: Features normalisieren (z-Standardisierung):

`x_norm = (x - mean(x)) / std(x)`

Danach hat `x_norm` Mittelwert ≈ 0 und Standardabweichung ≈ 1. Die Verlustlandschaft wird "kreisförmiger" und der Gradientenabstieg konvergiert deutlich schneller.

In [ ]:
x_mean = np.mean(x)
x_std  = np.std(x)
x_norm = (x - x_mean) / x_std

print(f'x (original): Mittelwert = {x_mean:.2f}, Std = {x_std:.2f}')
print(f'x_norm:       Mittelwert = {np.mean(x_norm):.4f}, Std = {np.std(x_norm):.4f}')

Jetzt haben wir die Funktionalität, um lineare Regression mittels Gradientenabstieg zu trainieren.

In [ ]:
def gradient_descent(x: np.ndarray, y: np.ndarray, learning_rate: float = 0.01,
                     epoch_count: int = 300, verbose: bool = True):
    """
    Trainiert ein lineares Regressionsmodell mit Gradientenabstieg.
    
    HINWEIS: x sollte normalisiert sein (z-Standardisierung), damit der
    Gradientenabstieg stabil und effizient konvergiert.
    
    :param x: Feature-Vektor als Array der Form `(n,)`; normalisiert.
    :param y: Label-Vektor als Array der Form `(n,)`.
    :param learning_rate: Lernrate (Schrittgrösse).
    :param epoch_count: Anzahl der Iterationen.
    :param verbose: Ob Zwischenergebnisse ausgegeben werden sollen.
    :return: Tupel `(w0, w1, MSE)`, wo `MSE` die Liste mit den MSE-Werten ist.
    """
    # Initialisierung mit Nullen
    w0 = 0.0
    w1 = 0.0
    mse_values = np.empty(epoch_count, dtype=np.float64)
    
    for index_epoch in range(epoch_count):
        # Gradienten berechnen
        dL_dw0, dL_dw1 = compute_gradients(x, y, w0, w1)
        
        # Parameter gleichzeitig aktualisieren
        w0 = w0 - learning_rate * dL_dw0
        w1 = w1 - learning_rate * dL_dw1
        
        # MSE berechnen und speichern
        current_mse = mse_loss(x, y, w0, w1)
        mse_values[index_epoch] = current_mse
        
        # Zwischenergebnisse ausgeben
        if verbose and \
            ((index_epoch + 1) % 50 == 0 or index_epoch == epoch_count - 1):
            print(f'Epoche {index_epoch + 1:4d} | MSE = {current_mse:.4f}, ')

    return w0, w1, mse_values


print('\nTraining mit Gradientenabstieg auf normalisierten Features...')
print(f'(Lernrate: 0.01, Epochen: 300)\n')

w0_gd, w1_gd, errors = gradient_descent(
    x_norm, y, learning_rate=0.01, epoch_count=300)

mse_gd = mse_loss(x_norm, y, w0_gd, w1_gd)
print(f'Gradientenabstieg Ergebnis auf normalisierten Features:')
print_model_info(w0_gd, w1_gd, mse_gd)

## Übung

Experimentieren Sie mit der Lernrate. Trainieren Sie das Modell dreimal mit den Lernraten $0.1$, $0.01$ und $0.001$, `(epoch_count=300, verbose=False)`. Vergleichen Sie die finale MSE jeweils mit der analytischen Lösung.

Was beobachten Sie?
 - Welche Lernrate konvergiert am schnellsten?
 - Was passiert bei einer zu grossen Lernrate (z.B. 2.0)?
 - Was passiert bei einer zu kleinen Lernrate?

In [ ]:
w0_analyt_norm, w1_analyt_norm = analytical_solution(x_norm, y)
mse_analyt_norm = mse_loss(x_norm, y, w0_analyt_norm, w1_analyt_norm)
print('Analytische Lösung:')
print_model_info(w0_analyt_norm, w1_analyt_norm, mse_analyt_norm)

for learning_rate in [0.001, 0.01, 0.1, 1.0]:
    w0, w1, current_errors = gradient_descent(x_norm, y, learning_rate, verbose=False)
    print(f'Gradientenabstieg mit LR={learning_rate} hat MSE = {current_errors[-1]:.3f}')

## Ergebnisse visualisieren

Das Gradientenabstieg-Modell wurde auf $x_{norm}$ trainiert:

$\hat{y} = w_0^{gd} + w_1^{gd} x_{norm} = w_0^{gd} + w_1^{gd} \frac{x - \mu_x}{\sigma_x}$

Wir rechnen es auf die Originalskala zurück, um es im selben Plot darzustellen:

`w1_orig = w1_gd / x_std`

`w0_orig = w0_gd - w1_gd * x_mean / x_std`

In [ ]:
w0_gd_orig = w0_gd - w1_gd * x_mean / x_std
w1_gd_orig = w1_gd / x_std
print(f'GD-Parameter, auf Originalskala zurückgerechnet:')
print(f'  w0 = {w0_gd_orig:.4f}  (analytisch: {w0_analyt:.4f})')
print(f'  w1 = {w1_gd_orig:.4f}  (analytisch: {w1_analyt:.4f})')

### Die Analytische Lösung mit dem Gradientenabstieg vergleichen

In der ersten Visualisierung zeigen wir beide Regressionsgeraden und die Residuen zu den Datenpunkten.

In [ ]:
def plot_regression(ax, x, y, w0, w1, line_color: str):
    """
    Plottet einen Datensatz, die Vorhersage ein LR-Modell sowie Residuen.
    :param ax: Achsen-Objekt zum plotten.
    :param x: X-coordinaten des Datensatzes.
    :param y: Y-coordinaten des Datensatzes.
    :param w0: Achsenabschnitt des linearen Modells.
    :param w1: Steigung des linearen Modells.
    :param line_color: Farbe der Modell-linie.
    """
    # Datensatz einzeichnen
    ax.scatter(x, y, color='steelblue', alpha=0.7, edgecolors='white', s=40,
           label='Datenpunkte', zorder=3)
    
    # Residuen einzeichnen
    for xi, yi in zip(x, y):
        yi_hat = predict(xi, w0, w1)
        ax.plot([xi, xi], [yi, yi_hat], alpha=0.5, color='gray', linewidth=0.8)

    # Regressionsgerade einzeichnen
    label = f'ŷ = {w0:.2f} + {w1:.4f}·x'
    ax.axline((0, w0), slope=w1, color=line_color, linewidth=2, label=label)
    
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)


_, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=100, sharex='all', sharey='all')

# Linker Plot: Analytische Lösung
ax = axes[0]
line_color = 'crimson'
plot_regression(ax, x, y, w0_analyt, w1_analyt, line_color)
ax.set_xlabel('TV-Werbebudget (Tsd. USD)', fontsize=11)
ax.set_ylabel('Verkaufszahlen (Tsd. Einheiten)', fontsize=11)
ax.set_title(f'Analytische Lösung, MSE = {mse_analyt:.3f}', fontsize=12)

# Rechter Plot: Gradientenabstieg (zurückgerechnet auf Originalskala)
ax = axes[1]
line_color = 'darkorange'
plot_regression(ax, x, y, w0_gd_orig, w1_gd_orig, line_color)
ax.set_xlabel('TV-Werbebudget (Tsd. USD)', fontsize=11)
ax.set_title(f'Gradientenabstieg, MSE = {mse_gd:.3f}', fontsize=12)

plt.tight_layout()
plt.show()

### Konvergenzkurve

Im nächsten Diagramm wird die Konvergenzkurve des Gradientenabstiegs gezeigt.

In [ ]:
_, ax = plt.subplots(figsize=(8, 4), dpi=100)
ax.plot(errors, color='darkorange', linewidth=2, label='MSE (Gradientenabstieg)')
ax.axhline(y=mse_analyt, color='crimson', linewidth=1.5, linestyle='--',
           label=f'Optimaler MSE (analytisch) = {mse_analyt:.3f}')
ax.set(title='Konvergenz des Gradientenabstiegs', xlabel='Epoche', ylabel='MSE')
ax.set(xlim=[0, errors.size], yscale='log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Verlustlandschaft

Wir visualisieren $L(w_0, w_1)$ als 2D-Konturplot, um ein Gefühl dafür zu bekommen, wie der Gradientenabstieg durch die Landschaft wandert. Der Verlust ist eine quadratische Funktion in $w_0$ und $w_1$, also die Landschaft hat eine Paraboloid-Form.

In [ ]:
# Wertebereiche für w0 und w1 um die analytische Lösung auf x_norm herum
w0_norm_opt, w1_norm_opt = analytical_solution(x_norm, y)
w0_werte = np.linspace(w0_norm_opt - 5, w0_norm_opt + 5, 100)
w1_werte = np.linspace(w1_norm_opt - 3, w1_norm_opt + 3, 100)

# Gitter erzeugen
W0, W1 = np.meshgrid(w0_werte, w1_werte)

# MSE für jede Kombination berechnen
MSE_grid = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        MSE_grid[i, j] = mse_loss(x_norm, y, W0[i, j], W1[i, j])

figure, ax = plt.subplots(figsize=(7, 5), dpi=100)

# Konturplot (Höhenlinien)
levels = np.linspace(MSE_grid.min(), MSE_grid.min() + 30, 25)
contour = ax.contourf(W0, W1, MSE_grid, levels=levels, cmap='viridis')
figure.colorbar(contour, ax=ax, label='MSE')

# Optimales Minimum markieren
ax.scatter(w0_norm_opt, w1_norm_opt, color='red', s=100, zorder=5,
           marker='*', label=f'Optimum ({w0_norm_opt:.2f}, {w1_norm_opt:.4f})')

ax.set_xlabel('w₀ (Bias)', fontsize=12)
ax.set_ylabel('w₁ (Steigung)', fontsize=12)
ax.set_title('MSE-Verlustlandschaft L(w₀, w₁)', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Hausaufgabe

Bisher haben wir nur das TV-Budget als Feature verwendet. Probieren Sie nun das Radio-Budget als Feature.

1. Laden Sie die Radio-Spalte aus rohdaten (Index `1`).
2. Berechnen Sie die analytische Lösung für Radio -> Verkäufe.
3. Berechnen Sie den MSE.
4. Erstellen Sie einen Scatter-Plot mit der Regressionsgeraden.
5. Vergleichen Sie den MSE mit dem TV-Modell. Welcher Werbekanal ist ein besserer Prädiktor für Verkäufe?

In [ ]:
pass